# LLM Evaluation and Benchmarking

## Comprehensive Guide to Evaluating LLM Performance

In this notebook, you will learn:
1. **LLM Quality Metrics** - Faithfulness, relevance, coherence
2. **RAG Evaluation** - Context precision, recall, answer quality
3. **LLM-as-Judge** - Using LLMs to evaluate LLMs
4. **Benchmark Datasets** - Standard evaluation datasets
5. **Production Monitoring** - Continuous evaluation

**Estimated Time:** 3-4 hours

## Part 1: LLM Quality Metrics

In [ ]:
from typing import List, Dict
from dataclasses import dataclass
import numpy as np
from enum import Enum

@dataclass
class EvaluationResult:
    """Evaluation result for a single response"""
    faithfulness: float  # 0-1, grounded in context
    relevance: float     # 0-1, addresses question
    coherence: float     # 0-1, logically organized
    fluency: float       # 0-1, grammatically correct
    accuracy: float      # 0-1, factually correct
    overall_score: float
    feedback: str = ""


class LLMEvaluator:
    """
    Comprehensive LLM evaluator
    
    Metrics:
    - Faithfulness: Is the answer grounded in context?
    - Relevance: Does it address the question?
    - Coherence: Is it logically organized?
    - Fluency: Is it grammatically correct?
    - Accuracy: Is it factually correct?
    """
    
    def __init__(self, judge_llm=None):
        self.judge_llm = judge_llm
        self.evaluation_history = []
    
    def evaluate(self, question: str, answer: str, 
                 context: List[str] = None, 
                 ground_truth: str = None) -> EvaluationResult:
        """
        Evaluate LLM response
        
        Args:
            question: User question
            answer: LLM answer
            context: Retrieved context (for RAG)
            ground_truth: Expected answer (if available)
        """
        # Calculate metrics
        faithfulness = self._evaluate_faithfulness(answer, context) if context else 0.5
        relevance = self._evaluate_relevance(question, answer)
        coherence = self._evaluate_coherence(answer)
        fluency = self._evaluate_fluency(answer)
        accuracy = self._evaluate_accuracy(answer, ground_truth) if ground_truth else 0.5
        
        # Overall score (weighted average)
        weights = {
            'faithfulness': 0.25,
            'relevance': 0.25,
            'coherence': 0.15,
            'fluency': 0.15,
            'accuracy': 0.20
        }
        
        overall = (
            faithfulness * weights['faithfulness'] +
            relevance * weights['relevance'] +
            coherence * weights['coherence'] +
            fluency * weights['fluency'] +
            accuracy * weights['accuracy']
        )
        
        result = EvaluationResult(
            faithfulness=faithfulness,
            relevance=relevance,
            coherence=coherence,
            fluency=fluency,
            accuracy=accuracy,
            overall_score=overall
        )
        
        self.evaluation_history.append(result)
        
        return result
    
    def _evaluate_faithfulness(self, answer: str, context: List[str]) -> float:
        """
        Evaluate if answer is grounded in context
        
        Check for:
        - Contradictions with context
        - Unsupported claims
        - Hallucinations
        """
        if not context:
            return 0.5
        
        context_text = " ".join(context).lower()
        answer_words = set(answer.lower().split())
        
        # Simple heuristic: word overlap
        context_words = set(context_text.split())
        overlap = len(answer_words & context_words) / len(answer_words) if answer_words else 0
        
        # In production: use NLI model or LLM judge
        return min(1.0, overlap * 2)  # Scale overlap to 0-1
    
    def _evaluate_relevance(self, question: str, answer: str) -> float:
        """
        Evaluate if answer addresses question
        """
        question_words = set(question.lower().split())
        answer_words = set(answer.lower().split())
        
        # Remove stop words
        stop_words = {'what', 'is', 'the', 'a', 'an', 'how', 'why', 'when', 'where'}
        question_words -= stop_words
        
        # Check overlap
        overlap = len(question_words & answer_words) / len(question_words) if question_words else 0
        
        return min(1.0, overlap * 2)
    
    def _evaluate_coherence(self, answer: str) -> float:
        """
        Evaluate logical organization and flow
        """
        sentences = answer.replace('.', '.\n').split('\n')
        sentences = [s.strip() for s in sentences if s.strip()]
        
        if len(sentences) < 2:
            return 1.0  # Single sentence is always coherent
        
        # Check for transition words
        transitions = {'however', 'therefore', 'moreover', 'furthermore', 
                      'first', 'second', 'finally', 'in conclusion',
                      'لكن', 'لذلك', 'أيضاً', 'أولاً', 'ثانياً'}  # Arabic
        
        has_transitions = any(
            any(t in sentence.lower() for t in transitions)
            for sentence in sentences
        )
        
        # Check sentence length variance (too varied = less coherent)
        lengths = [len(s.split()) for s in sentences]
        length_std = np.std(lengths) if lengths else 0
        
        score = 0.5
        if has_transitions:
            score += 0.3
        if length_std < 5:
            score += 0.2
        
        return min(1.0, score)
    
    def _evaluate_fluency(self, answer: str) -> float:
        """
        Evaluate grammatical correctness
        """
        # Simple checks
        
        # Check for repeated words
        words = answer.lower().split()
        repeats = sum(1 for i in range(len(words)-1) if words[i] == words[i+1])
        repeat_penalty = repeats / len(words) if words else 0
        
        # Check for incomplete sentences (no verb)
        # In production: use grammar checker
        
        score = 1.0 - repeat_penalty
        return max(0.0, score)
    
    def _evaluate_accuracy(self, answer: str, ground_truth: str) -> float:
        """
        Evaluate factual accuracy against ground truth
        """
        if not ground_truth:
            return 0.5
        
        # Simple text similarity
        answer_words = set(answer.lower().split())
        truth_words = set(ground_truth.lower().split())
        
        # Jaccard similarity
        intersection = len(answer_words & truth_words)
        union = len(answer_words | truth_words)
        
        return intersection / union if union else 0
    
    def evaluate_batch(self, test_cases: List[Dict]) -> Dict[str, float]:
        """
        Evaluate batch of test cases
        
        Returns:
            Aggregate metrics
        """
        results = []
        
        for test_case in test_cases:
            result = self.evaluate(
                question=test_case['question'],
                answer=test_case['answer'],
                context=test_case.get('context'),
                ground_truth=test_case.get('ground_truth')
            )
            results.append(result)
        
        # Aggregate
        metrics = {
            'faithfulness': np.mean([r.faithfulness for r in results]),
            'relevance': np.mean([r.relevance for r in results]),
            'coherence': np.mean([r.coherence for r in results]),
            'fluency': np.mean([r.fluency for r in results]),
            'accuracy': np.mean([r.accuracy for r in results]),
            'overall': np.mean([r.overall_score for r in results]),
            'num_samples': len(results)
        }
        
        return metrics


print("✓ LLM Evaluator loaded")

In [ ]:
# Example usage
evaluator = LLMEvaluator()

test_case = {
    'question': 'What is machine learning?',
    'answer': 'Machine learning is a subset of AI that enables systems to learn from data.',
    'context': ['Machine learning is part of artificial intelligence.'],
    'ground_truth': 'Machine learning is a subset of artificial intelligence that learns from data.'
}

result = evaluator.evaluate(
    question=test_case['question'],
    answer=test_case['answer'],
    context=test_case['context'],
    ground_truth=test_case['ground_truth']
)

print("Evaluation Result:")
print(f"  Faithfulness: {result.faithfulness:.2f}")
print(f"  Relevance:    {result.relevance:.2f}")
print(f"  Coherence:    {result.coherence:.2f}")
print(f"  Fluency:      {result.fluency:.2f}")
print(f"  Accuracy:     {result.accuracy:.2f}")
print(f"  Overall:      {result.overall_score:.2f}")

## Part 2: RAG-Specific Evaluation

In [ ]:
class RAGEvaluator:
    """
    RAG-specific evaluation
    
    Metrics:
    - Context Precision: How much of context was useful?
    - Context Recall: Does context contain ground truth?
    - Answer Relevancy: Does answer address question?
    - Faithfulness: Is answer grounded in context?
    """
    
    def __init__(self):
        pass
    
    def evaluate_context_precision(self, context: List[str], 
                                   ground_truth: str) -> float:
        """
        Context Precision
        What fraction of ground truth is covered by context?
        """
        gt_words = set(ground_truth.lower().split())
        context_words = set(' '.join(context).lower().split())
        
        # Remove stop words
        stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were'}
        gt_words -= stop_words
        context_words -= stop_words
        
        overlap = len(gt_words & context_words)
        precision = overlap / len(gt_words) if gt_words else 0
        
        return precision
    
    def evaluate_context_recall(self, context: List[str], 
                                ground_truth: str) -> float:
        """
        Context Recall
        What fraction of context is relevant to ground truth?
        """
        gt_words = set(ground_truth.lower().split())
        context_words = set(' '.join(context).lower().split())
        
        overlap = len(gt_words & context_words)
        recall = overlap / len(context_words) if context_words else 0
        
        return recall
    
    def evaluate_answer_relevancy(self, question: str, 
                                  answer: str) -> float:
        """
        Answer Relevancy
        Does answer address the question?
        """
        q_words = set(question.lower().split())
        a_words = set(answer.lower().split())
        
        # Remove question words and stop words
        question_filter = {'what', 'is', 'the', 'a', 'an', 'how', 'why'}
        q_words -= question_filter
        
        overlap = len(q_words & a_words)
        relevancy = overlap / len(q_words) if q_words else 0
        
        return overlap / len(q_words) if q_words else 0
    
    def evaluate_faithfulness(self, answer: str, 
                            context: List[str]) -> float:
        """
        Faithfulness
        Is answer supported by context?
        """
        context_text = ' '.join(context).lower()
        answer_sentences = answer.replace('.', '.\n').split('\n')
        
        supported_sentences = 0
        
        for sentence in answer_sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
            
            # Check if sentence is supported by context
            # Simple: word overlap
            sent_words = set(sentence.split())
            context_words = set(context_text.split())
            
            overlap = len(sent_words & context_words)
            if overlap > len(sent_words) * 0.3:  # 30% overlap threshold
                supported_sentences += 1
        
        total_sentences = len([s for s in answer_sentences if s.strip()])
        faithfulness = supported_sentences / total_sentences if total_sentences else 0
        
        return faithfulness
    
    def evaluate_full(self, question: str, answer: str, 
                     context: List[str], 
                     ground_truth: str) -> Dict[str, float]:
        """
        Complete RAG evaluation
        """
        return {
            'context_precision': self.evaluate_context_precision(context, ground_truth),
            'context_recall': self.evaluate_context_recall(context, ground_truth),
            'answer_relevancy': self.evaluate_answer_relevancy(question, answer),
            'faithfulness': self.evaluate_faithfulness(answer, context)
        }


# Example
rag_evaluator = RAGEvaluator()

rag_test = {
    'question': 'What is deep learning?',
    'answer': 'Deep learning uses neural networks with multiple layers.',
    'context': [
        'Deep learning is a subset of machine learning.',
        'It uses neural networks with many layers.'
    ],
    'ground_truth': 'Deep learning is machine learning with neural networks.'
}

metrics = rag_evaluator.evaluate_full(
    question=rag_test['question'],
    answer=rag_test['answer'],
    context=rag_test['context'],
    ground_truth=rag_test['ground_truth']
)

print("\nRAG Evaluation Metrics:")
print("-" * 40)
for metric, value in metrics.items():
    print(f"{metric:20s}: {value:.2f}")

## Part 3: LLM-as-Judge

In [ ]:
class LLMJudge:
    """
    Use LLM to evaluate LLM responses
    
    Advantages:
    - No ground truth needed
    - Can evaluate complex criteria
    - Provides explanations
    
    Disadvantages:
    - Cost
    - Bias
    - Inconsistency
    """
    
    def __init__(self, llm):
        self.llm = llm
    
    def judge(self, question: str, answer: str, 
              criteria: List[str] = None) -> Dict:
        """
        Judge answer quality
        """
        if criteria is None:
            criteria = ['accuracy', 'relevance', 'completeness', 'clarity']
        
        # Build prompt
        prompt = self._build_judge_prompt(question, answer, criteria)
        
        # In production: call LLM
        # response = self.llm.generate(prompt)
        
        # Mock response
        response = {
            'scores': {c: 0.8 for c in criteria},
            'feedback': 'Good answer overall.'
        }
        
        return response
    
    def _build_judge_prompt(self, question: str, answer: str, 
                           criteria: List[str]) -> str:
        """
        Build judge prompt
        """
        criteria_text = "\n".join([f"- {c}" for c in criteria])
        
        prompt = f"""You are an expert evaluator. Rate the following answer.

Question: {question}

Answer: {answer}

Rate the answer on these criteria (0-1 scale):
{criteria_text}

Provide scores and brief feedback.

Format:
Scores:
- criterion1: score
- criterion2: score

Feedback: [your feedback]
"""
        
        return prompt
    
    def pairwise_comparison(self, question: str, 
                           answer_a str, answer_b: str) -> Dict:
        """
        Compare two answers
        """
        prompt = f"""Compare these two answers to the question.

Question: {question}

Answer A: {answer_a}

Answer B: {answer_b}

Which answer is better? Consider:
- Accuracy
- Completeness
- Clarity
- Relevance

Respond with: 'A' or 'B' or 'Tie'
Explain your reasoning.
"""
        
        # In production: call LLM
        # response = self.llm.generate(prompt)
        
        return {
            'winner': 'A',
            'reasoning': 'Answer A is more accurate and complete.'
        }


print("✓ LLM Judge ready")

## Part 4: Benchmark Datasets

In [ ]:
# Standard benchmark datasets for LLM evaluation

BENCHMARK_DATASETS = {
    # General Knowledge
    'MMLU': {
        'description': 'Multi-task Language Understanding',
        'tasks': 57,
        'format': 'Multiple choice',
        'url': 'https://github.com/hendrycks/test'
    },
    
    # Reasoning
    'BIG-Bench': {
        'description': 'Beyond the Imitation Game',
        'tasks': 200+,
        'format': 'Various',
        'url': 'https://github.com/google/BIG-bench'
    },
    
    # Reading Comprehension
    'SQuAD': {
        'description': 'Stanford Question Answering Dataset',
        'format': 'Extractive QA',
        'url': 'https://rajpurkar.github.io/SQuAD-explorer/'
    },
    
    # Common Sense
    'HellaSwag': {
        'description': 'Harder Endings, Longer contexts, Low-shot Activities',
        'format': 'Sentence completion',
        'url': 'https://rowanzellers.com/hellaswag/'
    },
    
    # Math
    'GSM8K': {
        'description': 'Grade School Math 8K',
        'format': 'Math word problems',
        'url': 'https://github.com/openai/grade-school-math'
    },
    
    # Code
    'HumanEval': {
        'description': 'OpenAI HumanEval',
        'format': 'Code generation',
        'url': 'https://github.com/openai/human-eval'
    },
    
    # Arabic
    'ArabicMMLU': {
        'description': 'Arabic Multi-task Language Understanding',
        'tasks': 'STEM, Social Sciences, Humanities',
        'format': 'Multiple choice',
        'url': 'https://github.com/inceptionai/arabic-mmlu'
    },
    
    'Alghafa': {
        'description': 'Arabic LLM Benchmark',
        'tasks': 7,
        'format': 'Various NLP tasks',
        'url': 'https://github.com/mbzuai-nlp/Alghafa'
    }
}

# Display benchmarks
print("LLM Benchmark Datasets")
print("=" * 60)

for name, info in BENCHMARK_DATASETS.items():
    print(f"\n{name}:")
    print(f"  Description: {info['description']}")
    if 'tasks' in info:
        print(f"  Tasks: {info['tasks']}")
    print(f"  Format: {info['format']}")
    print(f"  URL: {info['url']}")

## Part 5: Production Monitoring

In [ ]:
from datetime import datetime
import json

class ProductionMonitor:
    """
    Continuous evaluation in production
    
    Track:
    - Response quality over time
    - User satisfaction
    - Error rates
    - Latency
    - Cost
    """
    
    def __init__(self):
        self.metrics_history = []
        self.alerts = []
    
    def log_request(self, request_id: str, question: str, 
                   answer: str, latency_ms: float, 
                   tokens_used: int, cost: float):
        """
        Log production request
        """
        log_entry = {
            'timestamp': datetime.now().isoformat(),
            'request_id': request_id,
            'question': question,
            'answer': answer,
            'latency_ms': latency_ms,
            'tokens_used': tokens_used,
            'cost': cost,
            'quality_score': None  # To be filled by evaluation
        }
        
        self.metrics_history.append(log_entry)
    
    def calculate_metrics(self, time_window: str = '1h') -> Dict:
        """
        Calculate production metrics
        """
        if not self.metrics_history:
            return {}
        
        latencies = [m['latency_ms'] for m in self.metrics_history]
        costs = [m['cost'] for m in self.metrics_history]
        tokens = [m['tokens_used'] for m in self.metrics_history]
        
        metrics = {
            'total_requests': len(self.metrics_history),
            'avg_latency_ms': np.mean(latencies),
            'p95_latency_ms': np.percentile(latencies, 95),
            'p99_latency_ms': np.percentile(latencies, 99),
            'avg_cost': np.mean(costs),
            'total_cost': sum(costs),
            'avg_tokens': np.mean(tokens),
            'total_tokens': sum(tokens)
        }
        
        return metrics
    
    def check_alerts(self, thresholds: Dict) -> List[str]:
        """
        Check for alert conditions
        """
        alerts = []
        
        metrics = self.calculate_metrics()
        
        if 'latency_p95' in thresholds:
            if metrics.get('p95_latency_ms', 0) > thresholds['latency_p95']:
                alerts.append(f"High latency: {metrics['p95_latency_ms']:.0f}ms")
        
        if 'cost_per_hour' in thresholds:
            if metrics.get('total_cost', 0) > thresholds['cost_per_hour']:
                alerts.append(f"High cost: ${metrics['total_cost']:.2f}")
        
        self.alerts.extend(alerts)
        return alerts
    
    def export_dashboard_data(self) -> Dict:
        """
        Export data for dashboard
        """
        return {
            'metrics': self.calculate_metrics(),
            'alerts': self.alerts,
            'recent_requests': self.metrics_history[-10:],
            'timestamp': datetime.now().isoformat()
        }


# Example usage
monitor = ProductionMonitor()

# Log some requests
for i in range(100):
    monitor.log_request(
        request_id=f"req_{i}",
        question=f"Question {i}",
        answer=f"Answer {i}",
        latency_ms=np.random.normal(200, 50),
        tokens_used=np.random.randint(100, 500),
        cost=np.random.uniform(0.001, 0.01)
    )

# Calculate metrics
metrics = monitor.calculate_metrics()

print("\nProduction Metrics:")
print("-" * 40)
for metric, value in metrics.items():
    if isinstance(value, float):
        print(f"{metric:20s}: {value:.2f}")
    else:
        print(f"{metric:20s}: {value}")

# Check alerts
thresholds = {
    'latency_p95': 300,
    'cost_per_hour': 1.0
}

alerts = monitor.check_alerts(thresholds)
if alerts:
    print(f"\n⚠️  Alerts: {len(alerts)}")
    for alert in alerts:
        print(f"  - {alert}")
else:
    print("\n✓ No alerts")

## Summary: LLM Evaluation

In [ ]:
from IPython.display import Markdown

summary = """
## LLM Evaluation - Key Takeaways

### 1. Quality Metrics
- **Faithfulness**: Grounded in context (no hallucinations)
- **Relevance**: Addresses the question
- **Coherence**: Logically organized
- **Fluency**: Grammatically correct
- **Accuracy**: Factually correct

### 2. RAG Metrics
- **Context Precision**: How much context was useful?
- **Context Recall**: Does context contain ground truth?
- **Answer Relevancy**: Does answer address question?
- **Faithfulness**: Is answer supported by context?

### 3. LLM-as-Judge
- Use LLM to evaluate LLM responses
- No ground truth needed
- Can evaluate complex criteria
- Watch for bias and inconsistency

### 4. Benchmarks
- **MMLU**: General knowledge (57 tasks)
- **BIG-Bench**: Reasoning (200+ tasks)
- **SQuAD**: Reading comprehension
- **GSM8K**: Math reasoning
- **HumanEval**: Code generation
- **ArabicMMLU**: Arabic knowledge

### 5. Production Monitoring
- Track latency (p95, p99)
- Monitor costs
- Set up alerts
- Continuous evaluation
- User feedback loop

### Best Practices
1. Evaluate before deployment
2. Monitor continuously in production
3. Use multiple metrics
4. Combine automated + human evaluation
5. Track trends over time
6. Set up alerting for regressions
"""

Markdown(summary)